# 141 남은 미성형 추가 전문가: 학습 정상 꼬리와 동시변화 확인
139 개발후보를 고정한 후 intensity4LR의 innernormal99/99.5와 방향확인조건75/90/95를 비교. 추가전문가 선택은 outerRun 제외 innerTP>=1 FP<=2 조건. 기존139규칙은 이전개발결과로고른것이므로전체독립신규평가와구분.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables';ns={}
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\nval=lambda n:dense[n].to_numpy(float)\nvs=lambda n:val(n+'__past10_std')\nvd=lambda n:val(n+'__delta1')\nve=lambda n:val(n+'__dev10')\nthermal={\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\n}\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\n# Collapse several opposite-direction disturbances into shared intensity/count features.\njointbase=np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Hopper_Temperature'),vs('Max_Injection_Speed'),vs('Cushion_Position'),vs('Plasticizing_Time'),vs('Barrel_Temperature_5'),vs('Average_Back_Pressure')])\nmemo={}\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n yy=y[ref];normal=yy==0\n assert set(g[ref]).isdisjoint(g[q])\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(jointbase[ref][normal])\n a=imp.transform(jointbase[ref]);b=imp.transform(jointbase[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n def features(x):\n  j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))\n  return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])\n model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(features(a),yy)\n memo[key]=model.predict_proba(features(b))[:,1]\n return memo[key]\n",ns)
lab=ns['lab'];y=ns['y'];g=ns['g'];use=ns['use'];dense=ns['dense']
base=pd.read_csv(t/'rg3_shortshot_139_predictions.csv',parse_dates=['TimeStamp'])
base=base[base.model.eq('primary_three_seed_mean')].set_index('TimeStamp')
gates={
 'opposed_cold_hot':np.minimum(-dense.Hopper_Temperature__dev10,dense.Barrel_Temperature_5__delta1).to_numpy(),
 'hopper_cool':-dense.Hopper_Temperature__dev10.to_numpy(),
 'barrel3_rise':dense.Barrel_Temperature_3__delta1.to_numpy(),
 'hopper_stability':-dense.Hopper_Temperature__past10_std.to_numpy()
}
records=[];choices=[];candidates=[]
def stats(yy,pr):
 tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
 return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
for block in pd.unique(g):
 ref=np.flatnonzero((g!=block)&use);q=np.flatnonzero(g==block)
 outer=ns['predict'](ref,q);inner=np.full(len(ref),np.nan)
 for held in pd.unique(g[ref]):
  mask=g[ref]==held;inner[mask]=ns['predict'](ref[~mask],ref[mask])
 old=base.loc[lab.TimeStamp.iloc[q],'prediction'].to_numpy(bool)
 options=[]
 for budget in [.005,.01]:
  cut=float(np.quantile(inner[y[ref]==0],1-budget,method='higher'))
  for feature,values in gates.items():
   normal_values=values[ref][(y[ref]==0)&np.isfinite(values[ref])]
   for quant in [.75,.9,.95]:
    gatecut=float(np.quantile(normal_values,quant,method='higher'))
    pi=(inner>=cut)&(values[ref]>=gatecut);pq=(outer>=cut)&(values[q]>=gatecut)
    met=stats(y[ref],pi);tag=feature+'_'+str(quant)+'_budget'+str(budget)
    candidates.append(dict(block=block,model=tag,innerTP=met[0],innerFP=met[1],innerF1=met[4],score_threshold=cut,gate_threshold=gatecut))
    options.append((tag,met,pq))
    for j,i in enumerate(q):records.append(dict(model=tag,TimeStamp=lab.TimeStamp.iloc[i],run_id=block,type=lab.type.iloc[i],actual=int(y[i]),prediction=int(old[j]|pq[j]),baseline=int(old[j]),new=int(pq[j]),score=float(outer[j])))
 valid=[o for o in options if o[1][0]>=1 and o[1][1]<=2]
 best=max(valid,key=lambda o:(o[1][4],-o[1][1])) if valid else None
 choices.append(dict(block=block,selected='none' if best is None else best[0]))
 pq=np.zeros(len(q),bool) if best is None else best[2]
 for j,i in enumerate(q):records.append(dict(model='primary_inner_addition',TimeStamp=lab.TimeStamp.iloc[i],run_id=block,type=lab.type.iloc[i],actual=int(y[i]),prediction=int(old[j]|pq[j]),baseline=int(old[j]),new=int(pq[j]),score=float(outer[j])))
p=pd.DataFrame(records);rows=[]
for name,z in p.groupby('model'):
 z=z[z.type.isin(['정상','미성형'])]
 tp,fp,fn,tn,f=stats(z.actual,z.prediction)
 rows.append(dict(model=name,TP=tp,FP=fp,FN=fn,TN=tn,F1=f))
m=pd.DataFrame(rows).sort_values('F1',ascending=False)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'choices'),(pd.DataFrame(candidates),'inner_candidates')]:obj.to_csv(t/f'rg3_shortshot_141_{name}.csv',index=False,encoding='utf-8-sig')
print(m.head(10).to_string(index=False))
print(m[m.model.eq('primary_inner_addition')].to_string(index=False))
print(pd.DataFrame(choices).to_string(index=False))


                            model  TP  FP  FN  TN       F1
     hopper_cool_0.95_budget0.005   6  11   2 555 0.480000
      hopper_cool_0.95_budget0.01   6  11   2 555 0.480000
      hopper_cool_0.9_budget0.005   6  12   2 554 0.461538
    barrel3_rise_0.75_budget0.005   5   9   3 557 0.454545
    barrel3_rise_0.95_budget0.005   5   9   3 557 0.454545
hopper_stability_0.95_budget0.005   5   9   3 557 0.454545
hopper_stability_0.75_budget0.005   5   9   3 557 0.454545
     barrel3_rise_0.9_budget0.005   5   9   3 557 0.454545
 hopper_stability_0.9_budget0.005   5   9   3 557 0.454545
opposed_cold_hot_0.95_budget0.005   6  13   2 553 0.444444
                 model  TP  FP  FN  TN  F1
primary_inner_addition   5  12   3 554 0.4
 block                     selected
rg3_R1 hopper_cool_0.95_budget0.005
rg3_R2  hopper_cool_0.9_budget0.005
rg3_R3                         none
rg3_R4 hopper_cool_0.95_budget0.005
rg3_R5                         none
